In [18]:
# Load the libraries
import pandas as pd
import re
import os
from langdetect import detect, LangDetectException
import emoji
from collections import Counter

In [19]:
# Path
RAW_DIR = '../data/raw'
CLEAN_DIR = '../data/cleaned'
CHANNEL = 'sut_tw'

In [20]:
# Timeline 
TIMELINE_FILES = {
    'timeline_1': 'timeline_1_2025_Jun24_Dec28.csv',
    'timeline_2': 'timeline_2_2025_Dec29_2026_Feb28.csv',
    'timeline_3': 'timeline_3_2026_Mar01_May16.csv'
}

In [21]:
dfs = []
for label, fname in TIMELINE_FILES.items():
    path = os.path.join(RAW_DIR, fname)
    df = pd.read_csv(path, encoding = 'utf-8-sig')
    df['timeline'] = label
    dfs.append(df)
    print(f'Loaded {label}: {len(df)} messages')

raw = pd.concat(dfs, ignore_index = True)
print(f'\nTotal raw messages: {len(raw)}')

raw.head(2)

Loaded timeline_1: 8601 messages
Loaded timeline_2: 1269 messages
Loaded timeline_3: 2460 messages

Total raw messages: 12330


,message_id,date,text,timeline
0,129507,2025-12-28T14:30:36+00:00,تنها کسی که تو مملکت داره کارش رو خوب انجام می...,timeline_1
1,129512,2025-12-28T15:05:28+00:00,اولین سالی که بابانوئل برای بچه‌های ما چیزی آو...,timeline_1


In [ ]:
import emoji

def extract_tweet_text(text, channel=CHANNEL):
    '''
    Remove empty lines, the trailing @channel line,
    and the last line (writer name).
    Emojis are stripped first to handle emoji usernames.
    Returns only the tweet text.
    '''
    if pd.isna(text):
        return ''
    
    text = emoji.replace_emoji(str(text), replace='')
    
    lines = [line.strip() for line in text.split('\n')]
    
    non_empty = [line for line in lines if line]

    if non_empty and non_empty[-1].lower() == f'@{channel.lower()}':
        non_empty = non_empty[:-1]   # drop the channel tag
    
    if non_empty:
        non_empty = non_empty[:-1]   # drop the writer line
    
    return '\n'.join(non_empty).strip()

In [23]:
raw['cleaned_text'] = raw['text'].apply(extract_tweet_text)

print('Examples of cleaned text')
for i in range(min(3, len(raw))):
    print(f'--Original--\n{raw.loc[i, 'text']}\n--Cleaned--\n{raw.loc[i, 'cleaned_text']}\n')

Examples of cleaned text
--Original--
تنها کسی که تو مملکت داره کارش رو خوب انجام میده چسب زن الویه نامینو!
داداش یکم شل بگیر باز نمیشه این
^mahdi^
@sut_tw
--Cleaned--
تنها کسی که تو مملکت داره کارش رو خوب انجام میده چسب زن الویه نامینو!
داداش یکم شل بگیر باز نمیشه این

--Original--
اولین سالی که بابانوئل برای بچه‌های ما چیزی آورد، فکر می‌کنید واکنش دخترم چی بود؟
صبح بیدار شد و هدیه‌ش رو پای درخت دید، با نگرانی برگشت گفت وای مامان سنتا از کجا اومده تو؟ نکنه آقا دزده هم از همونجا بتونه بیاد:|
ذهن بچه‌م کاملا ایرانی بود!
^خانوم اف ام^
@sut_tw
--Cleaned--
اولین سالی که بابانوئل برای بچه‌های ما چیزی آورد، فکر می‌کنید واکنش دخترم چی بود؟
صبح بیدار شد و هدیه‌ش رو پای درخت دید، با نگرانی برگشت گفت وای مامان سنتا از کجا اومده تو؟ نکنه آقا دزده هم از همونجا بتونه بیاد:|
ذهن بچه‌م کاملا ایرانی بود!

--Original--
داشتم گالریمو مرتب می‌کردم عکس بچگی این تخم سگو دیدم که یه روز رندوم تصمیم گرفت دیگه نیاد پیشم 
😭
😭
خامه اگه این توییتو می‌بینی برگرد میوزاده 
^⁦☆^
@sut_tw
--Cleaned--
داشتم گالریمو مرتب

In [24]:
# Check if there is any username left in the tweets
problematic  = []

for idx, row in raw.iterrows():
    text = row['cleaned_text']
    if not isinstance(text, str) or text.strip() == '':
        continue

    lines_with_caret = [line for line in text.split('\n') if re.match(r'^\^[^\n]+\^$', line.strip())]

    mentions = re.findall(r'@\w+', text)

    if lines_with_caret or mentions:
        problematic.append({
            'message_id': row['message_id'],
            'timeline': row['timeline'],
            'caret_lines': lines_with_caret,
            'mentions': mentions,
            'text': text
        })

if problematic:
    print(f'Found {len(problematic)} messages with username')
else:
    print('No remaining username left in the dataset, ready to proceed!')

Found 219 messages with username


In [25]:
# Remove the problematic tweets for they are not many
bad_ids = [item['message_id'] for item in problematic]
before_drop = len(raw)
raw = raw[~raw['message_id'].isin(bad_ids)]
after_drop = len(raw)

print(f'Dropped {before_drop -  after_drop} messages with remaining usernames.')
print(f'Remaining messages {after_drop}')

Dropped 219 messages with remaining usernames.
Remaining messages 12111


In [26]:
# Check for any duplicates 
before = len(raw)

raw = raw.drop_duplicates(subset = 'message_id', keep = 'first')

raw = raw.drop_duplicates(subset = 'cleaned_text', keep = 'first')

after = len(raw)

print(f'Removed {before - after} duplicates messages')

Removed 695 duplicates messages


In [27]:
# Deep clean, remove URLS and Emojis
def deep_clean(text):
    if not isinstance(text, str):
        return ''
    
    # Remove URLS
    text = re.sub(r'https?://\S+|www\.\S+|t\.me/\S+', '', text)
    # @mentions 
    text = re.sub(r'@\w+', '', text)
    # Hashtags
    text = re.sub(r'#(\w+)', r'\1', text)
    # Emoji
    text = emoji.replace_emoji(text, replace = '')

    return text

In [28]:
raw['cleaned_text'] = raw['cleaned_text'].apply(deep_clean)

# Remove any row that now is empty
empty_mask = raw['cleaned_text'].str.strip() == ''
print(f'Removed {empty_mask.sum()} messages that were empty')
raw = raw[~empty_mask].copy()
print(f'Remaining Messages: {len(raw)}')

Removed 1 messages that were empty
Remaining Messages: 11415


In [29]:
# Keep only the Persian text
def is_persian(text):
    if len(text) < 10:
        return True
    try:
        return detect(text) == 'fa'
    except LangDetectException:
        return True

In [30]:
persian_mask = raw['cleaned_text'].apply(is_persian)
removed = (~persian_mask).sum()
print(f'Removed {removed} non-Persian messages')
raw = raw[persian_mask].copy()
print(f'Remaining messages: {len(raw)}')

Removed 175 non-Persian messages
Remaining messages: 11240


In [31]:
# Final check to save the cleaned dataset
TEXT_COL = 'cleaned_text'

# Basic info
total = len(raw)
empty = raw[TEXT_COL].isna().sum() + (raw[TEXT_COL].str.strip() == '').sum()
print(f'Total massages: {total}')
print(f'Empty messages: {empty}')

# Emoji
def has_emoji(text):
    return bool(emoji.emoji_count(str(text))) if isinstance(text, str) else False

emoji_mask = raw[TEXT_COL].apply(has_emoji)
emoji_count = emoji_mask.sum()
print(f'Messages with emoji: {emoji_count}')

# URLS
url_pattern = re.compile(r'https?://\S+|www\.\S+|t\.me/\S+')
url_mask = raw[TEXT_COL].apply(lambda x: bool(url_pattern.search(str(x))) if isinstance(x, str) else False)
print(f'Messages with URL: {url_mask.sum()}')

# Mentions 
mention_mask = raw[TEXT_COL].apply(lambda x: bool(re.search(r'@\w+', str(x))) if isinstance(x, str) else False)
print(f'Messages with mentions: {mention_mask.sum()}')

# Remaining usernames
def has_caret_line(text):
    if not isinstance(text, str):
        return False
    for line in text.split('\n'):
        if re.match(r'^\^[^\n]+\^$', line.strip()):
            return True
    return False

caret_mask = raw[TEXT_COL].apply(has_caret_line)
print(f'Messages with remaining ^write^: {caret_mask.sum()}')

# Language check
def detect_lang(text):
    if not isinstance(text, str) or len(text.strip()) < 10:
        return 'short'
    try:
        return detect(text)
    except LangDetectException:
        return 'und'
    
langs = raw[TEXT_COL].apply(detect_lang)
non_persian = langs[~langs.isin(['fa', 'short'])].count()
print(f'Messages in other languages: {non_persian}')

# Short texts
word_counts = raw[TEXT_COL].apply(lambda x: len(str(x).split()) if isinstance(x, str) else 0)
very_short = (word_counts < 2) & (word_counts > 0)
print(f'Very short messages: {very_short.sum()}')

# Duplicates 
duplicated = raw.duplicated(subset = TEXT_COL, keep = False)
print(f'Duplicated texts: {duplicated.sum()}')

# Per time-line stats
for tl in sorted(raw['timeline'].unique()):
    tl_df = raw[raw['timeline'] == tl]
    print(f'{tl}: {len(tl_df)} messages, avg words: {tl_df[TEXT_COL].apply(lambda x: len(str(x).split())).mean():.1f}')

# Summary
all_clean = (emoji_count == 0 and url_mask.sum() == 0 and mention_mask.sum() == 0 and caret_mask.sum() == 0 and non_persian == 0 and empty == 0 and very_short.sum() == 0)
if all_clean:
    print('Dataset is clean')
else:
    print('There are issues')

Total massages: 11240
Empty messages: 0
Messages with emoji: 0
Messages with URL: 0
Messages with mentions: 0
Messages with remaining ^write^: 0
Messages in other languages: 27
Very short messages: 42
Duplicated texts: 4
timeline_1: 7677 messages, avg words: 26.5
timeline_2: 1193 messages, avg words: 24.5
timeline_3: 2370 messages, avg words: 24.2
There are issues


In [32]:
# We remove the remaining issues
before = len(raw)

# Remove non-Persian messages
persian_mask = langs.isin(['fa', 'short'])
raw = raw[persian_mask].copy()

# Remove very short messages
raw = raw[~((word_counts < 2) & (word_counts > 0))].copy()

# Remove duplicated messages
raw = raw.drop_duplicates(subset = 'cleaned_text', keep = 'first')

after = len(raw)

print(f'Dropped {before - after} messages, Remaining: {after}')

Dropped 66 messages, Remaining: 11174


/tmp/ipykernel_194614/2049979872.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  raw = raw[~((word_counts < 2) & (word_counts > 0))].copy()


In [33]:
# Now we are ready to save the cleaned dataset
raw.reset_index(drop = True, inplace = True)
print(f'Final row count: {len(raw)}')

raw[['message_id', 'cleaned_text', 'timeline']].to_csv(
    os.path.join(CLEAN_DIR, 'all_cleaned.csv'),
    index = False,
    encoding = 'utf-8-sig'
)

print('Dataset cleaned and ready.')

Final row count: 11174
Dataset cleaned and ready.
